# Etapa 6. Calibração e validação do ky

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/06_calibracao_ky.ipynb)

O objetivo é encontrar os valores de **ky** de cada fase que fazem a quebra simulada pelo déficit hídrico
reproduzir a quebra observada.

### Por que não calibrar contra a média do IBGE

A produtividade real é PR = PA − (perda por manejo). Se o ky for ajustado para que a PA reproduza a média do IBGE,
ele "aprende" também o efeito do manejo, e o yield gap de manejo some por construção. É um raciocínio
**circular**. A solução é calibrar contra **quebras relativas** em situações em que o manejo varia pouco:

- **Estratégia A, envoltória superior:** em cada ano, o percentil 90 dos municípios (quem maneja bem), dividido
  pela tendência desse percentil.
- **Estratégia B, anomalia sem tendência:** em cada município, a produtividade dividida pelo "teto" da sua própria
  série, que é a tendência do quantil 0,9 (regressão quantílica).

Em ambas, o lado simulado é a **eficiência climática** EC = PA/PP, que depende só do ky e dos ISNA.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# No Colab, esta célula:
#   1) instala as bibliotecas que o Colab não tem;
#   2) copia o projeto do GitHub (dados + funções) para a máquina do Colab;
#   3) conecta o seu Google Drive, onde os resultados ficam guardados entre uma sessão e outra.
# Fora do Colab (no computador do orientador), ela só acerta as pastas.
import os
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if NO_COLAB:
    !pip -q install agrometeorologiapy==1.0.1
    if Path("/content/milho_oeste").exists():
        !git -C /content/milho_oeste pull -q     # já copiado nesta sessão: só atualiza
    else:
        !git clone -q https://github.com/fcoliveira-utfpr/milho_oeste.git /content/milho_oeste
    os.chdir("/content/milho_oeste")             # entra na pasta do projeto
    from google.colab import drive
    drive.mount("/content/drive")                # pede autorização na primeira vez
    SAIDA = Path("/content/drive/MyDrive/milho_oeste/resultados")
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    SAIDA = Path("resultados")

SAIDA.mkdir(parents=True, exist_ok=True)         # cria a pasta de resultados, se não existir
sys.path.append("src")                           # permite importar as funções do projeto (pasta src/)
print("Pasta do projeto:", Path.cwd())
print("Resultados em   :", SAIDA)

In [ ]:
import json
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings

import statsmodels.formula.api as smf
from scipy.optimize import minimize

import mza_oeste as M
from avaliacao import avaliar, rmse

warnings.filterwarnings("ignore", message="Maximum number of iterations")   # aviso da regressão quantílica

sim = pd.read_parquet(SAIDA / "simulacao_mza_oeste.parquet")
ibge = pd.read_csv("dados/ibge_pam_milho_oeste.csv")
ibge = ibge[(ibge["safra"] == 2) & (ibge["area_colhida_ha"] >= 100)]     # descarta áreas muito pequenas

prep = M.preparar_janela(sim)        # ISNA e PP das 4 datas da janela, organizados para contas rápidas
print("Município-anos simulados:", len(prep["indice"]))

## 1. Produtividade relativa observada

### Estratégia B (por município)

A **regressão quantílica** no quantil 0,9 ajusta uma reta que fica acima de 90% dos pontos: é o teto da série
naquele nível tecnológico. Municípios com menos de 15 anos de dados ficam de fora.

In [ ]:
def teto(df):
    modelo = smf.quantreg("rendimento_kg_ha ~ ano", df).fit(q=0.9, max_iter=5000)
    return df.assign(teto=modelo.predict(df))

n_anos = ibge.groupby("cod_ibge")["ano"].transform("count")
# um ajuste por município: o laço percorre os grupos e concat junta as tabelas de volta
obs = pd.concat([teto(g) for _, g in ibge[n_anos >= 15].groupby("cod_ibge")])
obs["rel_B"] = (obs["rendimento_kg_ha"] / obs["teto"]).clip(upper=1.0)
obs_B = obs.set_index(["cod_ibge", "ano"])["rel_B"]
print(f"{obs_B.index.get_level_values(0).nunique()} municípios; {len(obs_B)} município-anos")

# o teto de um município, para visualizar
exemplo = obs[obs["cod_ibge"] == obs["cod_ibge"].iloc[0]]
plt.plot(exemplo["ano"], exemplo["rendimento_kg_ha"], "o-", label="IBGE")
plt.plot(exemplo["ano"], exemplo["teto"], "--", label="teto (quantil 0,9)")
plt.legend(); plt.ylabel("kg/ha"); plt.show()

### Estratégia A (regional)

O percentil 90 de cada ano, dividido pela tendência (quantil 0,9) da própria série do percentil 90.

In [ ]:
p90 = ibge.groupby("ano")["rendimento_kg_ha"].quantile(0.90).rename("p90").reset_index()
p90["teto"] = smf.quantreg("p90 ~ ano", p90).fit(q=0.9, max_iter=5000).predict(p90)
obs_A = (p90.set_index("ano")["p90"] / p90.set_index("ano")["teto"]).clip(upper=1.0).rename("rel_A")
obs_A.round(2).to_frame().T

## 2. O lado simulado

`M.ec_janela(prep, ky)` devolve a EC de cada município-ano, com as 4 datas da janela.

**Atenção a um detalhe que decide o resultado.** O "teto" do IBGE é a produtividade dos **anos bons**, e mesmo
nos anos bons houve algum déficit hídrico. Então a produtividade relativa observada é a EC de cada ano **em
relação à EC dos anos bons**, e não a EC absoluta. Para comparar coisas iguais, o lado simulado é normalizado do
mesmo jeito: a EC de cada ano é dividida pelo quantil 0,9 da EC daquele município. Sem essa normalização, o
otimizador empurra todos os ky para o limite inferior (teste: troque `sim_B` por `M.ec_janela(prep, ky_de(x))` e
veja o que acontece).

Para a estratégia A, usa-se a média dos municípios em cada ano, normalizada da mesma forma.

Com 23 anos, ajustar os cinco ky ao mesmo tempo dá **sobreajuste**. Calibram-se os três que mais pesam
(vegetativo, floração e enchimento); estabelecimento e maturação ficam nos valores de Doorenbos & Kassam.

In [ ]:
def ky_de(x):
    """Monta o dicionário de ky a partir dos 3 valores calibrados (vegetativo, floração, enchimento)."""
    return {**M.KY_DK, "vegetativo": x[0], "floracao": x[1], "enchimento": x[2]}

ANOS_IBGE = range(2003, 2026)

def normalizar(ec, grupo):
    """Divide a EC pelo seu quantil 0,9 dentro de cada grupo (município) ou na série toda (grupo=None)."""
    ref = ec.quantile(0.9) if grupo is None else ec.groupby(level=grupo).transform("quantile", 0.9)
    return (ec / ref).clip(upper=1.0)

def sim_B(x):
    ec = M.ec_janela(prep, ky_de(x))
    ec = ec[ec.index.get_level_values("ano").isin(ANOS_IBGE)]       # só os anos com dado do IBGE
    return normalizar(ec, "cod_ibge")

def sim_A(x):
    ec = M.ec_janela(prep, ky_de(x)).groupby(level="ano").mean()
    return normalizar(ec[ec.index.isin(ANOS_IBGE)], None)

def erro(x, estrategia, anos=None):
    """RMSE entre a produtividade relativa observada e a simulada, só nos anos indicados."""
    o, s = (obs_A, sim_A(x)) if estrategia == "A" else (obs_B, sim_B(x))
    o, s = o.align(s, join="inner")                     # mantém só os pares que existem nos dois lados
    if anos is not None:
        manter = o.index.get_level_values("ano").isin(anos)
        o, s = o[manter], s[manter]
    return rmse(o, s)

x_dk = [M.KY_DK["vegetativo"], M.KY_DK["floracao"], M.KY_DK["enchimento"]]
print(f"RMSE com o ky de Doorenbos & Kassam: A = {erro(x_dk, 'A'):.3f}   B = {erro(x_dk, 'B'):.3f}")
print("Leva alguns minutos: são 8.000 combinações na próxima célula.")

## 3. Busca em grade

Testa **todas** as combinações numa malha de valores (0,1 a 2,0 de 0,1 em 0,1: 20 × 20 × 20 = 8.000 combinações).
É lenta, mas permite **ver** a superfície do erro. `product` gera todas as combinações.

In [ ]:
grade = np.round(np.arange(0.1, 2.01, 0.1), 2)
linhas = []
for kv, kf, ke in product(grade, grade, grade):
    linhas.append((kv, kf, ke, erro([kv, kf, ke], "B")))
res = pd.DataFrame(linhas, columns=["vegetativo", "floracao", "enchimento", "rmse"]).sort_values("rmse")
res.head(10)

In [ ]:
# superfície do erro: floração × enchimento, com o ky do vegetativo fixo no melhor valor
melhor_veg = res.iloc[0]["vegetativo"]
sup = res[res["vegetativo"] == melhor_veg].pivot(index="floracao", columns="enchimento", values="rmse")
fig, ax = plt.subplots(figsize=(6.5, 5))
c = ax.contourf(sup.columns, sup.index, sup.values, levels=20, cmap="viridis_r")
fig.colorbar(c, label="RMSE")
ax.plot(res.iloc[0]["enchimento"], res.iloc[0]["floracao"], "r*", ms=14, label="mínimo")
ax.plot(M.KY_DK["enchimento"], M.KY_DK["floracao"], "wo", ms=8, label="Doorenbos & Kassam")
ax.set_xlabel("ky enchimento"); ax.set_ylabel("ky floração"); ax.legend()
ax.set_title(f"Estratégia B (ky vegetativo = {melhor_veg})")
fig.savefig(SAIDA / "fig_superficie_erro.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Otimização numérica

O mesmo problema resolvido com um **otimizador** (`scipy.optimize.minimize`), que parte dos valores tabelados e
caminha "morro abaixo" no erro, dentro de limites plausíveis (0,05 a 2,5). Se a grade e o otimizador chegarem ao
mesmo ponto, a calibração é confiável.

In [ ]:
LIMITES = [(0.05, 2.5)] * 3
otimos = {}
for estrategia in ["A", "B"]:
    r = minimize(erro, x0=x_dk, args=(estrategia,), bounds=LIMITES, method="L-BFGS-B")
    otimos[estrategia] = r.x
    print(f"Estratégia {estrategia}: ky veg = {r.x[0]:.2f}, flor = {r.x[1]:.2f}, ench = {r.x[2]:.2f}   RMSE = {r.fun:.3f}")
print("Melhor ponto da grade (B):", res.iloc[0][["vegetativo", "floracao", "enchimento"]].tolist())

## 5. Validação deixando um ano de fora

Para cada ano, o ky é calibrado com os **outros** anos e testado no ano retirado. É a validação mais honesta,
porque o que muda de verdade de uma amostra para outra é o clima de cada ano. As colunas `ky_*` guardam o ky de
cada rodada: a dispersão delas é a **incerteza do ky**.

In [ ]:
anos = sorted(set(obs_B.index.get_level_values("ano")) & set(obs_A.index))
validacao, kys = {"A": [], "B": []}, {"A": [], "B": []}
for estrategia in ["A", "B"]:
    for ano in anos:
        treino = [a for a in anos if a != ano]
        x = minimize(erro, x0=x_dk, args=(estrategia, treino), bounds=LIMITES, method="L-BFGS-B").x
        o, s = (obs_A, sim_A(x)) if estrategia == "A" else (obs_B, sim_B(x))
        o, s = o.align(s, join="inner")
        teste = o.index.get_level_values("ano") == ano
        validacao[estrategia].append(pd.DataFrame({"obs": o[teste], "sim": s[teste], "ano": ano}))
        kys[estrategia].append({"ano": ano, "vegetativo": x[0], "floracao": x[1], "enchimento": x[2]})
    validacao[estrategia] = pd.concat(validacao[estrategia])
    kys[estrategia] = pd.DataFrame(kys[estrategia]).set_index("ano")
kys["B"].describe().round(2)

In [ ]:
# desempenho na validação, comparado com o ky tabelado (que não usa dados para ser ajustado)
tabela = {}
for estrategia in ["A", "B"]:
    v = validacao[estrategia]
    o, s_dk = (obs_A, sim_A(x_dk)) if estrategia == "A" else (obs_B, sim_B(x_dk))
    o, s_dk = o.align(s_dk, join="inner")
    tabela[f"{estrategia}, ky tabelado"] = avaliar(o, s_dk)
    tabela[f"{estrategia}, ky calibrado (validação)"] = avaliar(v["obs"], v["sim"])
pd.DataFrame(tabela).T.round(3)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 5))
for ax, estrategia in zip(axs, ["A", "B"]):
    v = validacao[estrategia]
    ax.scatter(v["obs"], v["sim"], s=12 if estrategia == "B" else 40, alpha=0.5)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)                 # linha 1:1
    ax.set_xlabel("Produtividade relativa observada")
    ax.set_ylabel("Eficiência climática simulada (PA/PP)")
    ax.set_title(f"Estratégia {estrategia}, validação")
fig.savefig(SAIDA / "fig_dispersao_validacao.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Tabela final e gravação

Com o orientador, escolha qual estratégia é a principal. O ky final é a calibração com **todos** os anos; o
intervalo vem das rodadas da validação.

In [ ]:
tab_ky = pd.DataFrame({"Doorenbos & Kassam": M.KY_DK})
for estrategia in ["A", "B"]:
    tab_ky[f"Estratégia {estrategia}"] = pd.Series(ky_de(otimos[estrategia]))
    faixa = kys[estrategia].quantile([0.05, 0.95])
    tab_ky[f"{estrategia}: faixa 5-95%"] = [f"{faixa.loc[0.05, f]:.2f}-{faixa.loc[0.95, f]:.2f}" if f in faixa else "fixo"
                                           for f in M.FASES]
tab_ky.round(2)

In [ ]:
# >>> DECISÃO: estratégia principal <<<
PRINCIPAL = "B"
KY_CAL = {f: round(float(v), 3) for f, v in ky_de(otimos[PRINCIPAL]).items()}

with open(SAIDA / "ky_calibrado.json", "w") as f:
    json.dump({"estrategia": PRINCIPAL, "ky": KY_CAL,
               "ky_A": {f: round(float(v), 3) for f, v in ky_de(otimos["A"]).items()},
               "ky_B": {f: round(float(v), 3) for f, v in ky_de(otimos["B"]).items()}}, f, indent=2)
tab_ky.to_csv(SAIDA / "tabela_ky.csv")
print("Gravado:", KY_CAL)

### Exercício

1. Os ky calibrados ficaram maiores ou menores que os de Doorenbos & Kassam? Em qual fase a diferença é maior?
   Compare com da Silva et al. (2019), que calibraram o ky do milho safrinha em Dourados (MS).
2. As estratégias A e B concordam? Se não, qual parece mais confiável, e por quê? (pense no número de pontos e no
   que cada uma representa)
3. A superfície do erro tem um mínimo bem definido ou um "vale" comprido? O que isso diz sobre a precisão do ky?

### O que escrever

No Material e Métodos, a subseção **Calibração e validação do coeficiente ky**: o argumento do raciocínio
circular, as equações das duas estratégias, a busca em grade, o otimizador e a validação. Nos Resultados, a
tabela do ky, a tabela de desempenho e as figuras da superfície do erro e da dispersão.